# 🇩🇪 LernSathi — Your German AI Tutor · Google Colab Edition

This notebook runs your **entire local AI-tutor codebase** in the cloud — nothing is changed, every file below is an exact copy of the repo:

| Stage | Model | Local file |
|---|---|---|
| 🎤 Speech → Text | OpenAI **Whisper `small`** (German) | `ai/speech/stt.py` |
| 💬 Conversation | **Qwen3 1.7B** via **Ollama** | `ai/llm/model.py` |
| 🔊 Text → Speech | **Piper** · Thorsten (de, medium) | `ai/speech/tts.py` |
| 🖥️ UI | **Streamlit** chat + browser mic | `ui/`, `app.py` |
| 🔗 Glue | `ConversationService` | `services/conversation_service.py` |

### How to use
1. **Runtime ▸ Change runtime type ▸ T4 GPU** *(recommended — works CPU-only too, just slower)*
2. **Runtime ▸ Run all** — total ≈ 8–12 min (most of it is one-time model downloads)
3. At the end you get a public **`https://…trycloudflare.com`** link → open it on any device, **allow the microphone**, pick your German level (**A1–C2**), and start speaking German 🗣️

> The Streamlit app streams through an HTTPS Cloudflare tunnel, so **voice input (microphone) works**, exactly like running locally.

In [ ]:
#@title Step 1 — System setup: Ollama · ffmpeg · cloudflared

# Install required system packages FIRST
!apt-get -qq update
!apt-get -qq install -y zstd ffmpeg

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# Verify installations
!echo "--- installed ---"
!ollama --version
!ffmpeg -version 2>/dev/null | head -n 1
!cloudflared --version

In [ ]:
# Start Ollama server in the background
!nohup ollama serve > /tmp/ollama.log 2>&1 &

# Give it a few seconds to start
!sleep 5

# Check the API
!curl http://127.0.0.1:11434/api/tags 

In [ ]:
#@title Step 2 — Install Python packages (~2 min)
# NOTE: this installs OpenAI's `openai-whisper`.
# (Do NOT `pip install whisper` — that's an unrelated time-series package.)
%pip install -q streamlit openai-whisper "piper-tts==1.7.0" ollama

import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

In [ ]:
#@title Step 3 — Recreate the project structure (`/content` = project root)
import pathlib

DIRS = [
    "ai/llm",
    "ai/speech",
    "services",
    "ui/mic_frontend",
    "models/tts",
    "audio/input",
    "audio/output",
]
FILES = [
    "ai/__init__.py",
    "ai/llm/__init__.py",
    "ai/speech/__init__.py",
    "services/__init__.py",
    "ui/__init__.py",
    "ai/llm/model.py",
    "ai/speech/stt.py",
    "ai/speech/tts.py",
    "services/conversation_service.py",
    "ui/chat.py",
    "ui/mic_widget.py",
    "ui/mic_frontend/index.html",
]

for d in DIRS:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
for f in FILES:
    pathlib.Path(f).touch(exist_ok=True)

print("Project tree ready ✔")

## 📦 The codebase (exact copies of your repo)

Every `%%writefile` cell below writes one file from your local project, byte-for-byte.

### 🧠 AI layer — STT / LLM / TTS

In [ ]:
%%writefile ai/llm/model.py
import ollama


_BASE_PROMPT = """
Du bist ein freundlicher und geduldiger deutscher Sprachlehrer.

Deine Aufgabe ist es, mit dem Benutzer auf Deutsch zu sprechen
und ihm dabei zu helfen, sein Deutsch zu verbessern.

Allgemeine Regeln:
- Sprich hauptsächlich auf Deutsch.
- Halte deine Antworten natürlich und nicht unnötig lang.
- Stelle gelegentlich Fragen, damit das Gespräch weitergeht.
- Wenn der Benutzer einen grammatikalischen Fehler macht,
  korrigiere ihn freundlich.
"""

LEVEL_PROMPTS = {
    "A1": _BASE_PROMPT + """
Das aktuelle Sprachniveau des Benutzers ist A1 (Anfänger).

Regeln für A1:
- Verwende nur SEHR einfache Wörter und kurze Sätze
  (höchstens 8 Wörter pro Satz).
- Benutze fast ausschließlich das Präsens.
- Sprich über einfache Themen: Begrüßung, Familie, Essen,
  Hobbys, Zahlen, Farben.
- Stelle einfache Ja/Nein- oder kurze W-Fragen.
- Erkläre Korrekturen kurz und einfach auf Englisch.
- Schreibe nie mehr als 2–3 kurze Sätze pro Antwort.
""",
    "A2": _BASE_PROMPT + """
Das aktuelle Sprachniveau des Benutzers ist A2 (Grundlagen).

Regeln für A2:
- Verwende häufige Alltagswörter und einfache Sätze.
- Benutze Präsens, Perfekt und das Präteritum von
  sein/haben/modalen Verben.
- Sprich über Alltagsthemen: Einkaufen, Reisen, Wetter,
  Arbeit, Termine, Wohnung.
- Baue einfache Nebensätze mit „weil", „dass" oder „wenn" ein.
- Erkläre Korrekturen kurz auf Englisch oder einfachem Deutsch.
- Schreibe höchstens 3–4 Sätze pro Antwort.
""",
    "B1": _BASE_PROMPT + """
Das aktuelle Sprachniveau des Benutzers ist B1 (Mittelstufe).

Regeln für B1:
- Sprich natürlich über Meinungen, Erfahrungen, Pläne und Träume.
- Benutze komplexere Strukturen: Nebensätze, Wechselpräpositionen
  und den Konjunktiv II für Höflichkeit.
- Führe gelegentlich neue, nützliche Wörter ein.
- Erkläre Korrekturen überwiegend auf Deutsch, bei Bedarf auf Englisch.
- Schreibe höchstens 4–5 Sätze pro Antwort.
""",
    "B2": _BASE_PROMPT + """
Das aktuelle Sprachniveau des Benutzers ist B2 (Fortgeschritten).

Regeln für B2:
- Diskutiere abstrakte und komplexe Themen: Medien, Umwelt,
  Kultur, Beruf, Gesellschaft.
- Argumentiere klar und benutze Passiv, Konjunktiv II und
  komplexere Satzstrukturen.
- Führe idiomatische Ausdrücke ein und erkläre sie kurz.
- Gib präzises Grammatik-Feedback auf Deutsch.
- Schreibe höchstens 5–6 Sätze pro Antwort.
""",
    "C1": _BASE_PROMPT + """
Das aktuelle Sprachniveau des Benutzers ist C1 (sehr fortgeschritten).

Regeln für C1:
- Führe nuancierte Gespräche über anspruchsvolle Themen:
  Gesellschaft, Wissenschaft, Politik, Beruf, Kunst.
- Benutze eine reiche, idiomatische Ausdrucksweise und
  stilistische Varianten.
- Gib differenziertes Feedback zu Stil, Register und Nuancen.
- Antworte ausschließlich auf Deutsch.
- Halte die Antworten fließend und natürlich wie im echten Leben.
""",
    "C2": _BASE_PROMPT + """
Das aktuelle Sprachniveau des Benutzers ist C2 (fast muttersprachlich).

Regeln für C2:
- Sprich wie unter Muttersprachlern: rhetorisch gewandt,
  mit Ironie, Wortspielen und feinen Nuancen, wo es passt.
- Benutze anspruchsvolles Vokabular und Fachsprache passend zum Thema.
- Gib Feedback wie ein Muttersprachler: Stil, Register, Klang.
- Antworte ausschließlich auf Deutsch.
- Sei anspruchsvoll, aber immer respektvoll und ermutigend.
""",
}


class GermanChatbot:

    def __init__(self, model_name: str = "qwen3:4b", level: str = "A1"):
        self.model_name = model_name

        if level not in LEVEL_PROMPTS:
            raise ValueError(
                f"Unknown level '{level}'. "
                f"Choose one of: {', '.join(LEVEL_PROMPTS)}"
            )
        self.level = level

    @property
    def system_prompt(self) -> str:
        return LEVEL_PROMPTS[self.level]

    def set_level(self, level: str):
        if level not in LEVEL_PROMPTS:
            raise ValueError(
                f"Unknown level '{level}'. "
                f"Choose one of: {', '.join(LEVEL_PROMPTS)}"
            )
        self.level = level

    def generate_response(self, messages: list[dict]) -> str:

        conversation = [
            {
                "role": "system",
                "content": self.system_prompt
            }
        ]

        conversation.extend(messages)

        response = ollama.chat(
            model=self.model_name,
            messages=conversation
        )

        return response["message"]["content"].strip()


In [ ]:
%%writefile ai/speech/stt.py
import torch
import whisper

class SpeechToText:
    def __init__(self, model_name: str = "small"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.model = whisper.load_model(
            model_name,
            device=self.device
        )

    def transcribe(self, audio_path:str ) -> str:
        result = self.model.transcribe(
            audio_path,
            language="de",
            fp16=self.device == "cuda"
        )  

        text = result["text"]
        return text.strip()

In [ ]:
%%writefile ai/speech/tts.py
from pathlib import Path
import subprocess
import sys
import uuid

class TextToSpeech:
    def __init__(self, model_path=None):
        project_root = Path(__file__).resolve().parents[2]

        self.project_root = project_root

        if model_path is None:
            model_path = (
                project_root
                / "models"
                / "tts"
                / "de_DE-thorsten-medium.onnx"
            )

        self.model_path = Path(model_path)

        if not self.model_path.exists():
            raise FileNotFoundError(
                f"TTS model not found: {self.model_path}"
            )

    def synthesize(
        self,
        text: str,
        output_path=None,
    ) -> str:

        if output_path is None:
            filename = f"response_{uuid.uuid4().hex}.wav"
            output = (
                self.project_root
                / "audio"
                / "output"
                / filename
            )
        else:
            output = Path(output_path)

            # If a relative path is supplied, make it relative
            # to the project root rather than tests/
            if not output.is_absolute():
                output = self.project_root / output

        output.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        result = subprocess.run(
            [
                sys.executable,
                "-m",
                "piper",
                "--model",
                str(self.model_path),
                "--output_file",
                str(output),
            ],
            input=text.encode("utf-8"),
            capture_output=True,
        )

        if result.returncode != 0:
            error = result.stderr.decode(
                "utf-8",
                errors="replace",
            )

            raise RuntimeError(
                f"Piper TTS failed:\n{error}"
            )

        return str(output)

### 🔗 Service layer — the conversation pipeline

In [ ]:
%%writefile services/conversation_service.py
from pathlib import Path

from ai.speech.stt import SpeechToText
from ai.speech.tts import TextToSpeech
from ai.llm.model import GermanChatbot


class ConversationService:

    def __init__(self, level: str = "A1"):
        print("Loading AI models...")

        self.stt = SpeechToText("small")
        self.chatbot = GermanChatbot(model_name="qwen3:1.7b", level=level)
        self.tts = TextToSpeech()

        print("All AI models loaded.")

    @property
    def level(self) -> str:
        return self.chatbot.level

    def set_level(self, level: str):
        """Switch tutor level without reloading any models."""
        self.chatbot.set_level(level)

    def process_text(
        self,
        user_text: str,
        conversation_history: list[dict],
    ) -> dict:

        if not user_text.strip():
            raise ValueError("Please enter some text.")

        # Generate AI response using conversation_history as context
        # (read-only; service does not modify session state)
        ai_response = self.chatbot.generate_response(conversation_history)

        # Synthesize TTS
        audio_output = self.tts.synthesize(ai_response)

        return {
            "ai_response": ai_response,
            "audio_path": str(audio_output),
        }

    def process_audio(
        self,
        audio_path: str,
        conversation_history: list[dict],
    ) -> dict:

        # 1. Speech → Text
        user_text = self.stt.transcribe(audio_path)

        if not user_text:
            raise ValueError("Could not understand the audio.")

        # 2. Generate AI response using conversation_history as context
        ai_response = self.chatbot.generate_response(conversation_history)

        # 3. Text → Speech
        audio_output = self.tts.synthesize(ai_response)

        # 4. Return everything
        return {
            "user_text": user_text,
            "ai_response": ai_response,
            "audio_path": str(audio_output),
        }

    # --------------------------------------------------
    # Staged methods (used by the UI for step-by-step feedback)
    # TEXT never touches STT; VOICE always goes through Whisper.
    # --------------------------------------------------

    def transcribe(self, audio_path: str) -> str:
        """Stage 1 (voice only): Whisper speech-to-text."""
        return self.stt.transcribe(audio_path)

    def generate_reply(self, conversation_history: list[dict]) -> str:
        """Stage 2: Qwen3 response. History must already include the latest user message."""
        return self.chatbot.generate_response(conversation_history)

    def speak(self, text: str) -> str:
        """Stage 3: Piper text-to-speech."""
        return str(self.tts.synthesize(text))

### 🖥️ UI layer — Streamlit chat, mic widget, main app

In [ ]:
%%writefile ui/chat.py
import base64
import streamlit as st
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parents[1]


# --------------------------------------------------
# Global styles
# --------------------------------------------------

def _local_css():
    st.markdown("""
    <style>
        /* Centered, spacious content column */
        .block-container {
            max-width: 820px;
            padding-top: 1.2rem;
            padding-bottom: 7rem;
        }
        #MainMenu {visibility: hidden;}
        footer {visibility: hidden;}
        header[data-testid="stHeader"] {background: transparent;}

        /* Message typography */
        .stChatMessage {
            padding: 6px 4px !important;
            background: transparent !important;
        }
        .stChatMessage [data-testid="stMarkdownContainer"] p {
            font-size: 1rem;
            line-height: 1.55;
        }
        .msg-label {
            font-size: 0.72rem;
            font-weight: 600;
            letter-spacing: 0.04em;
            text-transform: uppercase;
            color: #8a8f98;
            margin: 0 0 2px 0;
        }

        /* Compact recorder widget */
        div[data-testid="stAudioInput"] > div {
            min-height: unset;
        }
        section[data-testid="stAudioInput"] {
            border: none !important;
        }

        /* Composer buttons round & tidy */
        div.stButton > button {
            border-radius: 12px;
            font-size: 1.05rem;
        }
        div.stButton > button[kind="primary"] {
            background: #10a37f;
            border-color: #10a37f;
        }
        div.stButton > button[kind="primary"]:hover {
            background: #0d8c6d;
            border-color: #0d8c6d;
        }
        div.stButton > button[kind="primary"]:disabled {
            opacity: 0.45;
        }

        /* Example chips on welcome screen */
        .chip-row {display: flex; flex-wrap: wrap; gap: 8px; justify-content: center; margin-top: 10px;}
    </style>
    """, unsafe_allow_html=True)


# --------------------------------------------------
# Single message
# --------------------------------------------------

def _autoplay_audio(path: str):
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    st.markdown(
        '<audio controls autoplay style="width:100%;">'
        f'<source src="data:audio/wav;base64,{b64}" type="audio/wav">'
        "</audio>",
        unsafe_allow_html=True,
    )


def _render_message(message: dict, idx: int, audio_history: dict, autoplay: bool = False):
    role = message["role"]

    if role == "user":
        with st.chat_message("user", avatar="👤"):
            st.markdown('<p class="msg-label">You</p>', unsafe_allow_html=True)
            st.markdown(message["content"])
    else:
        with st.chat_message("assistant", avatar="🇩🇪"):
            st.markdown('<p class="msg-label">LernSathi</p>', unsafe_allow_html=True)
            st.markdown(message["content"])
            audio_path = audio_history.get(idx)
            if audio_path and Path(audio_path).exists():
                if autoplay:
                    _autoplay_audio(audio_path)
                else:
                    with open(audio_path, "rb") as f:
                        st.audio(f.read(), format="audio/wav")


# --------------------------------------------------
# Levels (CEFR)
# --------------------------------------------------

LEVELS = {
    "A1": {"name": "Beginner", "desc": "First words & very simple sentences"},
    "A2": {"name": "Elementary", "desc": "Everyday small talk, simple past"},
    "B1": {"name": "Intermediate", "desc": "Opinions, plans & experiences"},
    "B2": {"name": "Upper-Intermediate", "desc": "Debates & abstract topics"},
    "C1": {"name": "Advanced", "desc": "Fluent, nuanced discussion"},
    "C2": {"name": "Proficient", "desc": "Near-native sophistication"},
}

EXAMPLES = {
    "A1": ["Hallo! Wie geht es dir?", "Ich heiße Anna.", "Ich lerne Deutsch."],
    "A2": ["Was machst du gern am Wochenende?", "Gestern war ich einkaufen.", "Wie ist das Wetter bei dir?"],
    "B1": ["Erzähl mir von deiner Stadt.", "Ich möchte meine Meinung üben.", "Was hast du letztes Jahr gemacht?"],
    "B2": ["Lass uns über soziale Medien diskutieren.", "Ist Fernsehen noch zeitgemäß?", "Welche Rolle spielt Kunst in deinem Leben?"],
    "C1": ["Wie beeinflusst KI die Arbeitswelt?", "Diskutieren wir über Bildungssysteme.", "Erkläre mir ein deutsches Idiom."],
    "C2": ["Ironie im Alltag – Fluch oder Segen?", "Deine Sicht auf moderne Literatur?", "Führe ein Bewerbungsgespräch mit mir."],
}


def render_level_select():
    st.markdown(
        """
        <div style="text-align:center; padding:56px 12px 8px;">
            <div style="font-size:3rem; line-height:1;">🇩🇪</div>
            <h1 style="font-size:1.9rem; font-weight:700; margin:14px 0 4px;">LernSathi</h1>
            <p style="color:#57606a; font-size:1.05rem; margin:0 0 6px;">
                Your German AI Tutor
            </p>
            <p style="color:#57606a; margin:0;">
                Choose your German level to begin.<br>
                The conversation adapts to what you pick.
            </p>
            <hr style="border:none; border-top:1px solid #e6e8eb; width:220px; margin:26px auto;">
        </div>
        """,
        unsafe_allow_html=True,
    )

    rows = [list(LEVELS.items())[i:i + 3] for i in range(0, len(LEVELS), 3)]
    for r, row in enumerate(rows):
        cols = st.columns(3)
        for c, (code, meta) in enumerate(row):
            with cols[c]:
                st.markdown(
                    f"""
                    <div style="text-align:center; padding:14px 6px 4px;">
                        <div style="font-size:1.5rem; font-weight:800;">{code}</div>
                        <div style="font-size:0.9rem; font-weight:600; margin-top:2px;">{meta['name']}</div>
                        <div style="color:#57606a; font-size:0.78rem; margin-top:4px; min-height:2.4em;">
                            {meta['desc']}
                        </div>
                    </div>
                    """,
                    unsafe_allow_html=True,
                )
                if st.button("Start", key=f"level_{code}", use_container_width=True,
                             type="primary" if (r * 3 + c) == 0 else "secondary"):
                    st.session_state.pending_level = code
                    st.rerun()


# --------------------------------------------------
# Welcome screen (English UI, level-aware examples)
# --------------------------------------------------

def _welcome_state(level: str):
    meta = LEVELS[level]
    phrases = EXAMPLES.get(level, [])

    st.markdown(
        f"""
        <div style="text-align:center; padding:56px 12px 8px;">
            <div style="font-size:3rem; line-height:1;">🇩🇪</div>
            <h1 style="font-size:1.9rem; font-weight:700; margin:14px 0 4px;">LernSathi</h1>
            <p style="color:#57606a; font-size:1.05rem; margin:0 0 10px;">
                Your German AI Tutor
            </p>
            <p style="margin:0;">
                <span style="display:inline-block; background:#e7f7f1; color:#0d8c6d;
                border-radius:999px; padding:4px 14px; font-weight:700; font-size:0.85rem;">
                    Level {level} · {meta["name"]}
                </span>
            </p>
            <p style="color:#57606a; margin:16px 0 0;">
                Practice German through natural conversation.<br>
                You can type or speak in German.<br>
                Don't worry about mistakes — LernSathi will help you.
            </p>
            <hr style="border:none; border-top:1px solid #e6e8eb; width:220px; margin:26px auto;">
            <p style="color:#8a8f98; font-size:0.85rem; margin:0 0 4px;">Try saying</p>
        </div>
        """,
        unsafe_allow_html=True,
    )

    cols = st.columns(len(phrases))
    for i, phrase in enumerate(phrases):
        with cols[i]:
            if st.button(phrase, key=f"phrase_{i}", use_container_width=True):
                st.session_state.chat_text_input = phrase
                st.rerun()


# --------------------------------------------------
# Public entry point
# --------------------------------------------------

def render_chat(messages: list, is_processing: bool, audio_history: dict,
                level: str, autoplay_idx: int | None = None):
    _local_css()

    if not messages:
        _welcome_state(level)
        return

    for idx, msg in enumerate(messages):
        _render_message(msg, idx, audio_history, autoplay=(idx == autoplay_idx))

    if is_processing:
        st.caption("⏳ Working on your message…")

In [ ]:
%%writefile ui/mic_widget.py
import base64
import os

import streamlit.components.v1 as components

_FRONTEND_DIR = os.path.join(os.path.dirname(__file__), "mic_frontend")

_mic_component = components.declare_component("lernsathi_mic", path=_FRONTEND_DIR)


def record_mic(action: str = "idle", key: str = "lernsathi_mic"):
    """
    Custom mic component. Starts recording as soon as it is rendered.

    action="stop" -> stops recording and returns {"bytes": wav, "id": int}
    Mic errors    -> returns {"error": str, "id": int}
    Otherwise     -> None (still recording)
    """
    val = _mic_component(action=action, key=key, default=None)
    if not val:
        return None
    if "b64" in val:
        return {"bytes": base64.b64decode(val["b64"]), "id": val["id"]}
    return val


In [ ]:
%%writefile ui/mic_frontend/index.html
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8" />
<style>
  html, body { margin: 0; padding: 0; background: transparent; }
</style>
</head>
<body>
<script>
(function () {
  "use strict";

  var started = false;
  var audioCtx = null;
  var sourceNode = null;
  var procNode = null;
  var mediaStream = null;
  var samples = [];
  var totalLen = 0;

  function send(type, data) {
    var msg = Object.assign({ isStreamlitMessage: true, type: type }, data || {});
    window.parent.postMessage(msg, "*");
  }

  function componentReady() {
    send("streamlit:componentReady", { apiVersion: 1 });
    send("streamlit:setFrameHeight", { height: 0 });
  }

  function setComponentValue(value) {
    send("streamlit:setComponentValue", { value: value, dataType: "json" });
  }

  function onRender(event) {
    if (!event.data || event.data.type !== "streamlit:render") return;
    var action = (event.data.args && event.data.args.action) || "idle";
    if (!started) {
      start();
    } else if (action === "stop") {
      stop(true);
    }
    // action === "idle" while recording -> keep going
  }

  async function start() {
    if (started) return;
    try {
      mediaStream = await navigator.mediaDevices.getUserMedia({
        audio: { echoCancellation: true, noiseSuppression: true },
      });
      audioCtx = new (window.AudioContext || window.webkitAudioContext)();
      if (audioCtx.state === "suspended") {
        try { await audioCtx.resume(); } catch (e) {}
      }
      sourceNode = audioCtx.createMediaStreamSource(mediaStream);
      procNode = audioCtx.createScriptProcessor(4096, 1, 1);
      samples = [];
      totalLen = 0;
      procNode.onaudioprocess = function (e) {
        var d = e.inputBuffer.getChannelData(0);
        samples.push(new Float32Array(d));
        totalLen += d.length;
      };
      sourceNode.connect(procNode);
      procNode.connect(audioCtx.destination);
      started = true;
    } catch (err) {
      setComponentValue({ error: String((err && err.message) || err), id: Date.now() });
    }
  }

  function mergeSamples() {
    var out = new Float32Array(totalLen);
    var off = 0;
    for (var i = 0; i < samples.length; i++) {
      out.set(samples[i], off);
      off += samples[i].length;
    }
    return out;
  }

  function encodeWav(s, rate) {
    var buf = new ArrayBuffer(44 + s.length * 2);
    var v = new DataView(buf);
    function ws(o, t) { for (var i = 0; i < t.length; i++) v.setUint8(o + i, t.charCodeAt(i)); }
    ws(0, "RIFF"); v.setUint32(4, 36 + s.length * 2, true);
    ws(8, "WAVE"); ws(12, "fmt ");
    v.setUint32(16, 16, true);
    v.setUint16(20, 1, true);
    v.setUint16(22, 1, true);
    v.setUint32(24, rate, true);
    v.setUint32(28, rate * 2, true);
    v.setUint16(32, 2, true);
    v.setUint16(34, 16, true);
    ws(36, "data"); v.setUint32(40, s.length * 2, true);
    var o = 44;
    for (var i = 0; i < s.length; i++) {
      var x = Math.max(-1, Math.min(1, s[i]));
      v.setInt16(o, x < 0 ? x * 0x8000 : x * 0x7fff, true);
      o += 2;
    }
    return buf;
  }

  function bufToB64(buf) {
    var u8 = new Uint8Array(buf), s = "", chunk = 0x8000;
    for (var i = 0; i < u8.length; i += chunk) {
      s += String.fromCharCode.apply(null, u8.subarray(i, i + chunk));
    }
    return btoa(s);
  }

  function stop(emit) {
    if (!started) return;
    started = false;
    var rate = audioCtx.sampleRate;
    try { procNode.disconnect(); } catch (e) {}
    try { sourceNode.disconnect(); } catch (e) {}
    try { mediaStream.getTracks().forEach(function (t) { t.stop(); }); } catch (e) {}
    try { audioCtx.close(); } catch (e) {}
    if (emit && totalLen > 0) {
      var wav = encodeWav(mergeSamples(), rate);
      setComponentValue({ b64: bufToB64(wav), id: Date.now(), sample_rate: rate });
    }
    samples = [];
    totalLen = 0;
  }

  window.addEventListener("message", onRender);
  window.addEventListener("load", componentReady);
})();
</script>
</body>
</html>


### ▶️ `app.py` — the Streamlit entry point

In [ ]:
%%writefile app.py
import os
import uuid
import streamlit as st

from ui.chat import render_chat, render_level_select, _local_css, LEVELS
from ui.mic_widget import record_mic
from services.conversation_service import ConversationService


# --------------------------------------------------
# Page configuration
# --------------------------------------------------

st.set_page_config(
    page_title="LernSathi",
    page_icon="🇩🇪",
    layout="wide",
)

_local_css()


# --------------------------------------------------
# Session state (deterministic across reruns)
# --------------------------------------------------

_defaults = {
    "level": None,             # selected CEFR level (A1–C2); None = not chosen yet
    "messages": [],            # single source of truth for the chat
    "audio_history": {},       # assistant msg index -> tts wav path
    "is_recording": False,     # mic capture in progress
    "mic_action": "idle",      # pending command for the mic component
    "last_mic_id": 0,          # dedupe guard for emitted recordings
    "is_processing": False,    # global pipeline lock
    "processing_stage": "",
}
for key, val in _defaults.items():
    if key not in st.session_state:
        st.session_state[key] = val

# Clear widget-bound keys BEFORE their widgets are instantiated this run
if st.session_state.pop("clear_chat_input", False):
    st.session_state.chat_text_input = ""


# --------------------------------------------------
# Load AI models once (Whisper small / qwen3:1.7b / Piper)
# --------------------------------------------------

@st.cache_resource
def load_service():
    return ConversationService()


service = load_service()

# Keep the cached service in sync with the selected level
if st.session_state.level is not None:
    service.set_level(st.session_state.level)


# --------------------------------------------------
# Helpers
# --------------------------------------------------

def paint():
    """Redraw the conversation from session state."""
    autoplay_idx = st.session_state.pop("autoplay_idx", None)
    render_chat(
        messages=st.session_state.messages,
        is_processing=st.session_state.is_processing,
        audio_history=st.session_state.audio_history,
        level=st.session_state.level,
        autoplay_idx=autoplay_idx,
    )


def reset_conversation():
    """Clear the chat history (keeps the selected level)."""
    st.session_state.messages = []
    st.session_state.audio_history = {}
    st.session_state.is_recording = False
    st.session_state.mic_action = "idle"
    st.session_state.is_processing = False
    st.session_state.processing_stage = ""
    st.session_state.chat_text_input = ""


def _fail(message: str, exc: Exception):
    st.error(f"{message} Please try again.")
    print(f"[LernSathi] {type(exc).__name__}: {exc}")


def _rollback_unpaired_user():
    """If a user message was added but no reply followed, remove it."""
    if st.session_state.messages and st.session_state.messages[-1]["role"] == "user":
        st.session_state.messages.pop()


def _finish():
    st.session_state.is_processing = False
    st.session_state.processing_stage = ""
    st.session_state.is_recording = False
    st.session_state.mic_action = "idle"
    st.session_state.clear_chat_input = True


def run_pipeline(user_text: str | None, audio_bytes: bytes | None):
    """
    One request -> one full pipeline run.

    TEXT :  LLM -> TTS                     (Whisper skipped entirely)
    VOICE:  Whisper -> LLM -> TTS          (transcription shown as the user bubble)
    """
    if st.session_state.is_processing:
        return

    st.session_state.is_processing = True

    try:
        # ---------- Stage 1a: transcription (voice only) ----------
        if audio_bytes is not None:
            rec_dir = os.path.join("audio", "input")
            os.makedirs(rec_dir, exist_ok=True)
            rec_path = os.path.join(rec_dir, f"user_recording_{uuid.uuid4().hex}.wav")
            with open(rec_path, "wb") as f:
                f.write(audio_bytes)

            with st.spinner("🎤 Transcribing your message…"):
                user_text = service.transcribe(rec_path)

            if not user_text or not user_text.strip():
                raise ValueError("The recording could not be understood.")

        if not user_text or not user_text.strip():
            raise ValueError("Empty message.")

        # User bubble is appended now but drawn on the post-rerun paint,
        # so the conversation never renders twice in one run.
        st.session_state.messages.append({"role": "user", "content": user_text})

        # ---------- Stage 2: tutor responds (text prepared) ----------
        with st.spinner("LernSathi is responding…"):
            reply = service.generate_reply(st.session_state.messages)

        # ---------- Stage 3: voice response (prepared BEFORE showing either) ----------
        with st.spinner("Preparing the voice response…"):
            audio_path = service.speak(reply)

        # Both ready -> shown together on the post-rerun paint (with autoplay)
        st.session_state.messages.append({"role": "assistant", "content": reply})
        last_idx = len(st.session_state.messages) - 1
        st.session_state.audio_history[last_idx] = audio_path
        st.session_state.autoplay_idx = last_idx

    except Exception as e:
        _rollback_unpaired_user()
        if isinstance(e, ValueError):
            _fail(str(e) + ".", e)
        elif "piper" in str(e).lower():
            _fail("I couldn't generate the voice response.", e)
        else:
            _fail("The tutor is temporarily unavailable.", e)

    finally:
        _finish()
        st.rerun()


# --------------------------------------------------
# Sidebar
# --------------------------------------------------

with st.sidebar:
    st.markdown("### 🇩🇪 LernSathi")
    st.caption("German AI Conversation Tutor")

    if st.button("＋ New conversation", use_container_width=True):
        reset_conversation()
        st.rerun()

    st.divider()

    st.markdown("**Current level**")
    if st.session_state.level is None:
        st.caption("Not selected yet")
    else:
        meta = LEVELS[st.session_state.level]
        st.markdown(
            f"<span style='display:inline-block; background:#e7f7f1; color:#0d8c6d;"
            f"border-radius:999px; padding:3px 12px; font-weight:700; font-size:0.85rem;'>"
            f"{st.session_state.level} · {meta['name']}</span>",
            unsafe_allow_html=True,
        )
        if st.button("🎚 Change level", use_container_width=True):
            reset_conversation()
            st.session_state.level = None
            st.rerun()

    st.divider()

    st.markdown("**Current session**")
    st.caption("German practice · not saved anywhere")
    st.markdown(f"**{len(st.session_state.messages)} messages** in this conversation")

    st.divider()

    st.markdown("**About**")
    st.caption(
        "Practice German through AI-powered "
        "conversation. Everything runs locally."
    )


# --------------------------------------------------
# Main area
# --------------------------------------------------

if st.session_state.level is None:
    # Level-first flow: no level chosen -> show picker, hide chat
    render_level_select()

    pending = st.session_state.pop("pending_level", None)
    if pending:
        st.session_state.level = pending
        service.set_level(pending)
        reset_conversation()
        st.rerun()
else:
    paint()


# --------------------------------------------------
# Composer  [ input ][ 🎤 / ✕ ][ ➤ ]
# idle      : [ input ][ 🎤 ][ ➤ ]   (➤ sends text)
# recording : [ input ][ ✕  ][ ➤ ]   (➤ stops & sends audio, ✕ discards)
# --------------------------------------------------

if st.session_state.level is not None:

    disabled = st.session_state.is_processing
    recording = st.session_state.is_recording and not disabled

    col_a, col_b, col_c = st.columns([8, 0.7, 0.7])

    with col_a:
        typed = st.text_input(
            "Message",
            placeholder="Write in German...",
            key="chat_text_input",
            label_visibility="collapsed",
            disabled=disabled,
        )

    with col_b:
        if recording:
            if st.button("✕", use_container_width=True, key="btn_cancel_rec"):
                st.session_state.is_recording = False
                st.session_state.mic_action = "idle"
                st.rerun()
        elif st.button("🎤", disabled=disabled, use_container_width=True, key="btn_mic"):
            st.session_state.is_recording = True
            st.rerun()

    with col_c:
        if recording:
            if st.button("➤", type="primary", use_container_width=True,
                         key="btn_send_rec"):
                st.session_state.mic_action = "stop"
                st.rerun()
        else:
            text_send_clicked = st.button(
                "➤", type="primary",
                disabled=disabled or not typed,
                use_container_width=True, key="btn_send",
            )

    # Pipeline runs OUTSIDE any column context so its spinners/errors
    # render full-width instead of squeezed under the send button.
    if not recording and text_send_clicked and typed and not disabled:
        run_pipeline(user_text=typed, audio_bytes=None)

    if recording:
        st.caption("🔴 Recording… press **➤** to send or **✕** to discard")

        result = record_mic(action=st.session_state.mic_action)
        st.session_state.mic_action = "idle"

        if isinstance(result, dict) and "error" in result:
            st.session_state.is_recording = False
            _fail(f"Microphone unavailable ({result['error']}).", Exception(result["error"]))
            st.rerun()
        elif isinstance(result, dict) and result.get("id") \
                and result["id"] != st.session_state.last_mic_id:
            st.session_state.last_mic_id = result["id"]
            st.session_state.is_recording = False
            run_pipeline(user_text=None, audio_bytes=result["bytes"])

In [ ]:
#@title Step 5 — Download the Piper voice · de_DE thorsten medium (~65 MB)
!mkdir -p models/tts
!wget -qc https://huggingface.co/rhasspy/piper-voices/resolve/main/de/de_DE/thorsten/medium/de_DE-thorsten-medium.onnx -O models/tts/de_DE-thorsten-medium.onnx
!wget -qc https://huggingface.co/rhasspy/piper-voices/resolve/main/de/de_DE/thorsten/medium/de_DE-thorsten-medium.onnx.json -O models/tts/de_DE-thorsten-medium.onnx.json
!ls -lh models/tts/

In [ ]:
#@title Step 6 — Start the Ollama server (background daemon)
import subprocess
import time
import urllib.request


def server_alive():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2)
        return True
    except Exception:
        return False


if server_alive():
    print("Ollama server already running ✔")
else:
    # start_new_session keeps the daemon alive after this cell finishes
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True,
    )
    for _ in range(60):
        if server_alive():
            break
        time.sleep(1)
    else:
        raise RuntimeError("Ollama server did not come up within 60 s")

print("Ollama server ready ✔  (http://127.0.0.1:11434)")

In [ ]:
#@title Step 7 — Pull the Qwen3 1.7B model into Ollama (~1.4 GB, one-time)
!ollama pull qwen3:1.7b
!ollama list

## ✅ Smoke test — hear the whole pipeline before opening the UI

This loads **Whisper small** (~460 MB, auto-download) and runs a full round-trip:
LLM reply → Piper voice → Whisper transcription of that voice.

In [ ]:
#@title Test the full pipeline: LLM → TTS → STT round-trip
from IPython.display import Audio, display

from ai.llm.model import LEVEL_PROMPTS
from services.conversation_service import ConversationService

LEVEL = "A1" #@param ["A1", "A2", "B1", "B2", "C1", "C2"]

print("Loading models (first run downloads Whisper small)...\n")
service = ConversationService(level=LEVEL)
print(f"Tutor level: {service.level}")

# 1) LLM — reply adapts to the selected level
reply = service.generate_reply([{"role": "user", "content": "Hallo, wer bist du?"}])
print("LernSathi >", reply, "\n")

# 2) Switch level on the fly — only the system prompt changes, no reload
service.set_level("C2")
reply_c2 = service.generate_reply([{"role": "user", "content": "Erzähl mir von deiner Stadt."}])
print(f"LernSathi [C2] >", reply_c2, "\n")
service.set_level(LEVEL)

# 3) TTS — play it right here
wav_path = service.speak("Hallo! Ich bin LernSathi, dein deutscher Sprachlehrer.")
print("TTS audio ->", wav_path)
display(Audio(wav_path))

# 4) STT — feed the generated audio back through Whisper
print("Whisper heard >", service.transcribe(wav_path))


In [ ]:
# #@title Optional — chat with the tutor directly in this notebook
# LEVEL = "A1" #@param ["A1", "A2", "B1", "B2", "C1", "C2"]

# if service.level != LEVEL:
#     service.set_level(LEVEL)   # instant switch, models stay loaded

# history = []
# print(f'Tutor level: {service.level}. Type in German. Type "quit" to exit.\n')
# while True:
#     try:
#         user = input("Du > ").strip()
#     except (KeyboardInterrupt, EOFError):
#         break
#     if not user or user.lower() in {"quit", "exit", "q"}:
#         break
#     history.append({"role": "user", "content": user})
#     reply = service.generate_reply(history)
#     history.append({"role": "assistant", "content": reply})
#     print(f"LernSathi > {reply}\n")


## 🌐 Launch the Streamlit UI (same experience as local)

The app starts on port 8501 and is exposed via a free Cloudflare quick tunnel.
**Keep the notebook tab open while demoing** — closing/disconnecting the runtime kills the tunnel.

In [ ]:
#@title Step 8 — Launch Streamlit + public HTTPS tunnel 🚀
import re
import subprocess
import time

# Clean up any previous instances
subprocess.run("pkill -f 'streamlit run' || true", shell=True)
subprocess.run("pkill -f cloudflared || true", shell=True)

st_log = open("streamlit.log", "w")
subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port=8501",
        "--server.address=127.0.0.1",
        "--server.headless=true",
        "--browser.gatherUsageStats=false",
    ],
    stdout=st_log,
    stderr=subprocess.STDOUT,
)
print("Streamlit starting on port 8501 ...")
time.sleep(8)

cf_log = open("cloudflared.log", "w")
subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8501", "--no-autoupdate"],
    stdout=cf_log,
    stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", open("cloudflared.log").read())
    if m:
        public_url = m.group(0)
        break

print()
if public_url:
    print("=" * 62)
    print(f"  🇩🇪 LernSathi is LIVE ->  {public_url}")
    print("=" * 62)
    print("Open the link, click 'Allow' when the browser asks for the")
    print("microphone, hit 🎤 and speak German!")
else:
    print("Could not find the tunnel URL yet. Log output:")
    print(open("cloudflared.log").read())

## 🛠️ Notes & Troubleshooting

| Symptom | Fix |
|---|---|
| First answer takes ~30–60 s | Normal — Qwen3 warms up on its first request; afterwards replies are fast |
| Microphone doesn't ask for permission | You must be on the `https://…trycloudflare.com` URL (not localhost); re-run **Step 8** to get a fresh tunnel |
| Tunnel stopped responding | The Colab VM idled out — re-run **Step 8** only (models stay loaded if the runtime wasn't deleted) |
| Everything broke / kernel restarted | Disk persists across kernel restarts: re-run **Step 6** onward. If the VM itself was recycled: **Runtime ▸ Run all** |
| No GPU available | Fine — Whisper falls back to CPU and Qwen3:1.7B runs on CPU too, just noticeably slower |
| Want a fresh conversation or another difficulty | Click **“＋ New conversation”** (same level) or **“🎚 Change level”** (back to A1–C2 picker) in the app sidebar |

**Demo tip:** run Steps 1–7 before your audience arrives; during the demo just fire Step 8, open the link, and talk. 🎤